# Fine-Tuning avec LoRA — BLOOMZ-560m sur English Quotes

**Objectif :** Fine-tuner le modèle `bigscience/bloomz-560m` sur le dataset `Abirate/english_quotes` en utilisant **LoRA** (Low-Rank Adaptation) via la bibliothèque PEFT.

**Étapes :**
1. Installation des bibliothèques nécessaires
2. Chargement du modèle pré-entraîné et du tokenizer
3. Chargement et prétraitement du dataset
4. Configuration de LoRA
5. Application de LoRA au modèle
6. Entraînement
7. Sauvegarde et rechargement du modèle
8. Génération de texte

## Étape 1 — Installation des bibliothèques

In [ ]:
# peft==0.4.0 est incompatible avec les versions récentes de transformers
# (PeftMixedModel n'existe pas dans 0.4.0)
# On installe une version récente compatible
%pip install -q "peft>=0.10.0"
!pip install -q datasets
!mkdir -p ../cache/working/peft_lab_outputs

## Étape 2 — Chargement du modèle et du tokenizer

On utilise `bigscience/bloomz-560m`, un modèle génératif multilingue de type **BLOOM** fine-tuné sur des tâches d'instruction (xP3 dataset).  
- Architecture : Transformer décodeur (causal LM)
- Taille : 560 millions de paramètres
- `AutoModelForCausalLM` charge automatiquement la bonne classe de modèle

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"

# Chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Chargement du modèle de base (fondation)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"Modèle chargé : {model_name}")
print(f"Nombre total de paramètres : {sum(p.numel() for p in foundation_model.parameters()):,}")

## Étape 3 — Chargement et prétraitement du dataset

Le dataset `Abirate/english_quotes` contient des citations célèbres en anglais avec leurs auteurs et tags.  
- On utilise **10% du split d'entraînement** (`split="train[:10%]"`)
- On tokenize la colonne `"quote"` pour préparer les séquences d'entraînement
- On affiche 5 exemples pour vérifier la structure

In [ ]:
from datasets import load_dataset

# Chargement et échantillonnage à 10%
data = load_dataset("Abirate/english_quotes", split="train[:10%]")

# Tokenisation des citations
data = data.map(
    lambda samples: tokenizer(samples["quote"]),
    batched=True
)

# Affichage de 5 exemples
train_sample = data.select(range(5))
display(train_sample.to_pandas()[["quote", "author", "tags"]])

## Étape 4 — Configuration de LoRA

**LoRA (Low-Rank Adaptation)** est une technique de fine-tuning efficace en paramètres (PEFT).  
Au lieu de mettre à jour tous les poids du modèle, LoRA ajoute de **petites matrices de bas rang** aux couches d'attention.

**Paramètres clés :**
| Paramètre | Valeur | Description |
|-----------|--------|-------------|
| `r` | 1 | Rang des matrices de décomposition (plus petit = moins de paramètres) |
| `lora_alpha` | 1 | Facteur d'échelle pour les mises à jour LoRA |
| `target_modules` | `["query_key_value"]` | Couches ciblées (projections QKV dans BLOOM) |
| `lora_dropout` | 0.05 | Régularisation pour éviter l'overfitting |
| `bias` | `"none"` | Les biais ne sont pas entraînés |
| `task_type` | `"CAUSAL_LM"` | Type de tâche : modèle de langage causal |

In [ ]:
import peft
from peft import LoraConfig, get_peft_model

# Configuration LoRA
lora_config = LoraConfig(
    r=1,                              # Rang des matrices LoRA
    lora_alpha=1,                     # Facteur d'échelle
    target_modules=["query_key_value"],  # Couches d'attention QKV de BLOOM
    lora_dropout=0.05,                # Dropout pour régularisation
    bias="none",                      # Les biais ne sont pas entraînés
    task_type="CAUSAL_LM"             # Tâche : génération de texte
)

print("Configuration LoRA :")
print(lora_config)

## Étape 5 — Application de LoRA au modèle de fondation

`get_peft_model()` enveloppe le modèle de fondation avec les adaptateurs LoRA.  
Seuls les paramètres LoRA (matrices de bas rang) seront entraînés.  
`print_trainable_parameters()` montre la réduction drastique du nombre de paramètres entraînables.

In [ ]:
# Application des adaptateurs LoRA au modèle de fondation
peft_model = get_peft_model(foundation_model, lora_config)

# Affichage du nombre de paramètres entraînables vs total
print(peft_model.print_trainable_parameters())

## Étape 6 — Configuration de l'entraînement

On utilise `TrainingArguments` et `Trainer` de HuggingFace pour gérer l'entraînement.

- `auto_find_batch_size=True` : ajuste automatiquement la taille de batch selon la mémoire disponible
- `learning_rate=3e-2` : taux d'apprentissage plus élevé que pour un fine-tuning complet (LoRA est plus stable)
- `num_train_epochs=1` : 1 époque suffit pour un test rapide
- `DataCollatorForLanguageModeling` : formate les batches pour la modélisation de langage causal (`mlm=False`)

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer
import os

output_directory = os.path.join("../cache/working", "peft_lab_outputs")
os.makedirs(output_directory, exist_ok=True)

training_args = TrainingArguments(
    report_to="none",                 # Pas de logging externe (W&B, etc.)
    output_dir=output_directory,
    auto_find_batch_size=True,        # Ajustement automatique du batch size
    learning_rate=3e-2,               # Taux d'apprentissage élevé (LoRA)
    num_train_epochs=1,               # 1 époque pour démonstration
    use_cpu=True                      # Forcer CPU (adapter si GPU disponible)
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,       # 5 exemples seulement (demo rapide)
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# Lancement de l'entraînement
print("Début de l'entraînement...")
trainer.train()
print("Entraînement terminé !")

## Étape 7 — Sauvegarde du modèle LoRA fine-tuné

On sauvegarde **uniquement les adaptateurs LoRA** (pas le modèle de fondation complet).  
`save_pretrained()` enregistre les poids LoRA + la configuration dans un dossier horodaté.  
Ce dossier peut ensuite être rechargé avec `PeftModel.from_pretrained()`.

In [ ]:
import time

# Horodatage pour nommer le dossier de sauvegarde
time_now = int(time.time())

peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")

# Sauvegarde des adaptateurs LoRA
trainer.model.save_pretrained(peft_model_path)

print(f"Modèle LoRA sauvegardé dans : {peft_model_path}")
print("Fichiers sauvegardés :", os.listdir(peft_model_path))

## Étape 8 — Rechargement du modèle pour l'inférence

Pour l'inférence, on recharge :
1. Le **modèle de fondation** (`foundation_model`)
2. Les **adaptateurs LoRA** sauvegardés via `PeftModel.from_pretrained()`

`is_trainable=False` indique qu'on est en mode inférence (pas d'entraînement supplémentaire).

In [ ]:
from peft import PeftModel

# Rechargement du modèle de fondation (sans LoRA)
loaded_model = AutoModelForCausalLM.from_pretrained(model_name)

# Application des adaptateurs LoRA sauvegardés sur le modèle de fondation
loaded_peft_model = PeftModel.from_pretrained(
    loaded_model,
    peft_model_path,
    is_trainable=False   # Mode inférence uniquement
)

print("Modèle PEFT rechargé avec succès !")
print(f"Type : {type(loaded_peft_model)}")

## Étape 9 — Génération de texte

On génère du texte à partir d'un prompt en utilisant le modèle fine-tuné.  
**Paramètres de génération :**
- `max_new_tokens=50` : longueur maximale de la séquence générée
- `do_sample=True` : active l'échantillonnage stochastique (plus créatif)
- `temperature=0.9` : contrôle la diversité (plus proche de 1 = plus aléatoire)
- `repetition_penalty=1.2` : pénalise les répétitions de tokens

In [ ]:
# Tokenisation du prompt
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")

# Génération de texte avec le modèle fine-tuné
outputs = loaded_peft_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=50,
    do_sample=True,
    temperature=0.9,
    repetition_penalty=1.2
)

# Décodage et affichage du texte généré
generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print("Texte généré :")
for text in generated_text:
    print(f"  >> {text}")

## Comparaison : modèle de base vs modèle fine-tuné

On compare la génération du modèle original (sans LoRA) avec le modèle fine-tuné sur les citations.

In [ ]:
prompt = "Two things are infinite: "
inputs = tokenizer(prompt, return_tensors="pt")

# Génération avec le modèle de FONDATION (sans LoRA)
with_no_lora = foundation_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=50,
    do_sample=True,
    temperature=0.9,
    repetition_penalty=1.2
)

# Génération avec le modèle FINE-TUNÉ (avec LoRA)
with_lora = loaded_peft_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=50,
    do_sample=True,
    temperature=0.9,
    repetition_penalty=1.2
)

print("=" * 60)
print("Modèle de fondation (sans LoRA) :")
print(tokenizer.batch_decode(with_no_lora, skip_special_tokens=True)[0])
print()
print("Modèle fine-tuné (avec LoRA) :")
print(tokenizer.batch_decode(with_lora, skip_special_tokens=True)[0])
print("=" * 60)

## Conclusion

Dans ce notebook, nous avons :

1. **Chargé** le modèle `bigscience/bloomz-560m` et son tokenizer
2. **Préparé** un échantillon de 10% du dataset `Abirate/english_quotes`
3. **Configuré LoRA** avec `r=1`, ciblant les couches `query_key_value` de BLOOM
4. **Fine-tuné** le modèle — seuls ~0.02% des paramètres ont été mis à jour
5. **Sauvegardé** les adaptateurs LoRA (fichier léger, pas le modèle entier)
6. **Rechargé** le modèle via `PeftModel.from_pretrained()`
7. **Généré** du texte à partir d'un prompt

**Avantages de LoRA :**
- Très peu de paramètres entraînables (économie mémoire et calcul)
- Le modèle de fondation reste intact
- Les adaptateurs peuvent être partagés et réutilisés facilement
- Compatible avec n'importe quel modèle Transformer